# Tự động sinh Caption cho Dataset Vỉa hè bằng LLM (Chạy trên GPU của Google Colab)
Notebook này sử dụng mô hình Qwen 2.5 (phiên bản 1.5B tham số) của HuggingFace, chạy trực tiếp trên T4 GPU của Colab để tối đa hóa tốc độ mà không cần API.

In [ ]:
!pip install -q transformers accelerate bitsandbytes tqdm

In [ ]:
# Kết nối với Google Drive của bạn
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import os
from tqdm.auto import tqdm
import torch
from transformers import pipeline

# Tải mô hình ngôn ngữ siêu tốc (Qwen 2.5 - 1.5B)
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
pipe = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.float16},
    device_map="auto",
)

In [ ]:
def generate_llm_caption(objects, severity, blocked):
    objects_str = ', '.join(objects) if objects else 'no specific obstacles'
    
    prompt = f"""
You are an AI generating image captions for an Urban Sidewalk Monitoring dataset.
Based strictly on the following detected attributes, write ONE concise, natural English sentence describing the scene.
- Detected objects on sidewalk: {objects_str}
- Violation severity: {severity}
- Is pedestrian path completely blocked? {blocked}

Output strictly the single sentence without any intro, outro, or quotes.
"""
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    
    # GPU xử lý sinh chữ
    outputs = pipe(
        messages,
        max_new_tokens=64,
        do_sample=False,
    )
    
    caption = outputs[0]["generated_text"][-1]['content'].strip().replace('\n', ' ')
    return caption

In [ ]:
# ĐƯỜNG DẪN FILE TRÊN GOOGLE DRIVE CỦA BẠN
json_path = '/content/drive/MyDrive/dataset_annotations/_annotations_merged.coco.json'
out_path = '/content/drive/MyDrive/dataset_annotations/_annotations_with_llm_captions.coco.json'

# Cơ chế tự động Resume: Nếu file out_path đã tồn tại, ta sẽ đọc tiếp từ đó để không phải chạy lại từ đầu
load_path = out_path if os.path.exists(out_path) else json_path

if not os.path.exists(load_path):
    print(f"\u274c LỖI: Không tìm thấy file tại {load_path}")
else:
    with open(load_path, 'r') as f:
        data = json.load(f)
        
    cat_id_to_name = {c['id']: c['name'].replace('_', ' ') for c in data['categories']}
    sidewalk_cat_ids = [c['id'] for c in data['categories'] if c['name'].lower() == 'sidewalk']
    ignore_names = ['sw2', 'sw3', 'sw4', 'sw5', 'sw6', 'My-Second-Project', 'My-Third-Project', 'object']
    ignore_cat_ids = [c['id'] for c in data['categories'] if c['name'] in ignore_names]
    
    img_to_anns = {img['id']: [] for img in data['images']}
    for ann in data['annotations']:
        img_to_anns[ann['image_id']].append(ann)
        
    print(f"Đang sinh Caption bằng GPU (Model: {model_id})...")
    
    images_to_process = data['images']
    save_interval = 100  # Cứ sau 100 ảnh sẽ lưu lại một lần vào Drive
    
    for i, img in enumerate(tqdm(images_to_process)):
        # Nếu ảnh này ĐÃ CÓ caption (do lần trước chạy chưa xong) thì bỏ qua, chạy tiếp ảnh mới
        if 'caption_llm' in img and img['caption_llm'] and not img['caption_llm'].startswith("Error"):
            continue
            
        anns = img_to_anns.get(img['id'], [])
        
        objects = []
        for a in anns:
            cid = a['category_id']
            if cid not in sidewalk_cat_ids and cid not in ignore_cat_ids:
                objects.append(cat_id_to_name[cid])
                
        severity = img.get('violation_severity', 'low')
        blocked = img.get('pedestrian_blocked', 'no')
        
        caption = generate_llm_caption(objects, severity, blocked)
        img['caption_llm'] = caption
        
        # Tự động lưu tiến trình định kỳ để chống mất dữ liệu khi rớt mạng/timeout
        if (i + 1) % save_interval == 0:
            with open(out_path, 'w') as f:
                json.dump(data, f)
                
    # Lưu lần cuối cùng khi hoàn tất 100%
    with open(out_path, 'w') as f:
        json.dump(data, f)
        
    print(f"\n\u2705 Hoàn tất! File kết quả đã được lưu an toàn tại Drive: {out_path}")